In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/07 16:09:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [7]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-07 14:11:42--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.239.238.133, 18.239.238.119, 18.239.238.152, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.239.238.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M   132MB/s    in 0.5s    

2026-03-07 14:11:42 (132 MB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



In [8]:
!wc -l yellow_tripdata_2025-11.parquet

282291 yellow_tripdata_2025-11.parquet


In [9]:
df = spark.read \
    .option("header", "true") \
    .parquet('yellow_tripdata_2025-11.parquet')

In [16]:
df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [14]:
df.schema

StructType([StructField('VendorID', IntegerType(), True), StructField('tpep_pickup_datetime', TimestampNTZType(), True), StructField('tpep_dropoff_datetime', TimestampNTZType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('Airport_fee', DoubleType(), True), StructField('cbd_congestio

In [26]:
# Might need to install pandas to confirm schema
# NB Long uses 8 bytes instead of 4 so would be more efficiant to correct schema

In [21]:
df = df.repartition(4)

In [22]:
df.write.parquet('yellow/2025')

In [3]:
df = spark.read.parquet('yellow/2025')

In [5]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [29]:
# Count trips on 15/11/25 - Only those that start on 15th

In [5]:
from pyspark.sql import functions as F

In [ ]:
# Functions needed
# to_date e.g. F.to_date(df.tpep_pickup_datetime))
# count_if e.g. df.select(F.count_if(sf.col('c2') % 2 == 0)).show()

In [11]:
df.filter(F.to_date("tpep_pickup_datetime") == "2025-11-15").count()

162604

In [16]:
df.select(F.count_if(F.to_date(df.tpep_pickup_datetime) == '2025-11-15')).show()

+------------------------------------------------------+
|count_if((to_date(tpep_pickup_datetime) = 2025-11-15))|
+------------------------------------------------------+
|                                                162604|
+------------------------------------------------------+



In [31]:
# What is the length of the longest trip in the dataset in hours?

In [17]:
# pyspark.sql.functions.timestamp_diff(unit, start, end)[source]
# Gets the difference between the timestamps in the specified units by truncating the fraction part.

# New in version 4.0.0.

# Parameters
# unitliteral string
# This indicates the units of the difference between the given timestamps. Supported options are (case insensitive): “YEAR”, “QUARTER”, “MONTH”, “WEEK”, “DAY”, “HOUR”, “MINUTE”, “SECOND”, “MILLISECOND” and “MICROSECOND”.

# startColumn or column name
# A timestamp which the expression subtracts from endTimestamp.

# endColumn or column name
# A timestamp from which the expression subtracts startTimestamp.

In [7]:
df.withColumn('length_in_hours', F.timestamp_diff("HOUR", df.tpep_pickup_datetime, df.tpep_dropoff_datetime)).show(1)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+---------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|length_in_hours|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+---------------+
|       2| 2025-11-07 18:37:45|  2025-11-07 18:41:51|              1|         0.78

In [6]:
df.select(
    F.max(
        F.timestamp_diff("HOUR", df.tpep_pickup_datetime, df.tpep_dropoff_datetime)
    )
).show()

[Stage 1:>                                                          (0 + 4) / 4]

+---------------------------------------------------------------------+
|max(timestampdiff(HOUR, tpep_pickup_datetime, tpep_dropoff_datetime))|
+---------------------------------------------------------------------+
|                                                                   90|
+---------------------------------------------------------------------+



In [12]:
df.select(
    F.round (
        F.max(
            F.timestamp_diff("SECOND", df.tpep_pickup_datetime, df.tpep_dropoff_datetime) / 3600
        ), 1
    )
).show()

[Stage 14:>                                                         (0 + 4) / 4]

+------------------------------------------------------------------------------------------+
|round(max((timestampdiff(SECOND, tpep_pickup_datetime, tpep_dropoff_datetime) / 3600)), 1)|
+------------------------------------------------------------------------------------------+
|                                                                                      90.6|
+------------------------------------------------------------------------------------------+



In [13]:
# Using the zone lookup data and the Yellow November 2025 data, what is the name of the LEAST frequent pickup location Zone?

In [14]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-07 16:36:30--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.239.238.119, 18.239.238.133, 18.239.238.212, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.239.238.119|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-07 16:36:30 (568 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [16]:
zones = spark.read \
    .option("header", "true") \
    .csv('taxi_zone_lookup.csv')

In [21]:
zones.show(200)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [19]:
df_zones = df.join(
    zones,
    df.PULocationID == zones.LocationID,
    "left"
)

result = (
    df_zones
    .groupBy("Zone")
    .count()
    .orderBy("count")   # ascending → least frequent first
)

result.show()

+--------------------+-----+
|                Zone|count|
+--------------------+-----+
|       Arden Heights|    1|
|Governor's Island...|    1|
|Eltingville/Annad...|    1|
|       Port Richmond|    3|
|         Great Kills|    4|
| Green-Wood Cemetery|    4|
|   Rossville/Woodrow|    4|
|       Rikers Island|    4|
|         Jamaica Bay|    5|
|         Westerleigh|   12|
|       West Brighton|   14|
|New Dorp/Midland ...|   14|
|             Oakwood|   14|
|        Crotona Park|   14|
|       Willets Point|   15|
|Breezy Point/Fort...|   16|
|Saint George/New ...|   17|
|       Broad Channel|   18|
|     Mariners Harbor|   21|
|Heartland Village...|   22|
+--------------------+-----+
only showing top 20 rows
